# BestShot — Chameleon Node Setup

Run this notebook from **Experiment > Jupyter Interface** on chameleoncloud.org.

Before running:
1. You must have an **active** `gpu_mi100` lease on CHI@TACC
2. Your SSH key must be uploaded to CHI@TACC

Run all cells top to bottom. At the end you will have a floating IP to SSH into.

## 1. Connect to your project

In [ ]:
from chi import server, context, lease, network
import os

context.version = "1.0"
context.choose_project()               # pick your CHI-XXXXXX project from the dropdown
context.choose_site(default="CHI@TACC")

## 2. Get your active lease

Change `YOUR_LEASE_NAME` to whatever you named your lease on Chameleon (e.g. `bestshot-serving`).

In [ ]:
LEASE_NAME = "serve_proj19_testing_lease"   # change this

l = lease.get_lease(LEASE_NAME)
l.show()   # status should say ACTIVE

## 3. Launch the bare metal server

Uses the ROCm-compatible Ubuntu image (required for AMD MI100 GPU).

**Note:** if you already have a server with this name in ERROR state, delete it first in the Horizon GUI before running this cell.

In [ ]:
username = os.getenv("USER")

s = server.Server(
    f"node-bestshot-{username}",
    reservation_id=l.node_reservations[0]["id"],
    image_name="CC-Ubuntu24.04-ROCm"
)
s.submit(idempotent=True)
print("Server submitted — waiting for it to become active...")

## 4. Configure security groups

In [ ]:
security_groups = [
    {"name": "allow-ssh", "port": 22, "description": "Enable SSH traffic on TCP port 22"},
    {"name": "allow-30283", "port": 30283, "description": "Enable Immich NodePort access on TCP port 30283"},
    {"name": "allow-30500", "port": 30500, "description": "Enable MLflow NodePort access on TCP port 30500"},
]

for sg in security_groups:
    secgroup = network.SecurityGroup({
        "name": sg["name"],
        "description": sg["description"],
    })
    secgroup.add_rule(direction="ingress", protocol="tcp", port=sg["port"])
    secgroup.submit(idempotent=True)
    s.add_security_group(sg["name"])

print(f"Updated security groups: {[sg['name'] for sg in security_groups]}")

## 5. Assign a floating IP so you can SSH in

In [ ]:
s.associate_floating_ip()

In [ ]:
s.refresh()
s.check_connectivity()

In [ ]:
s.refresh()
s.show(type="widget")   # note the floating IP in the Addresses row

## 5. Add team SSH keys

In [ ]:
s.execute("""cat >> ~/.ssh/authorized_keys << 'EOF'
# Anokhi
ssh-rsa AAAAB3NzaC1yc2EAAAADAQABAAABgQD2GL6pig+qGS9zhy/HvWs0CewQ5qqsg1QiG80Wxpko4URU4WeavVkmOl3iTPXYapXRbKvGXcDhGtgjO0OwYEfJK8Lhie86U4vavkcYn4bX1GsXcC6HWfUCu0rUVnMCDvq1pdifWb0fXO00OtaoImzg7vEi3BKltwzYJjaxSoG5xx+MEYniMh4lgIUF8nJ60xEX+GXn37TzglWkOFwRg+h7mhJN31BpZVpM1c3UT2MC1grwlbFo7YQl8NeKZiN9+HSBKokIBDO1QNximDnZ+Vl0ZgKojUSxmOAEvJYSio5I/wmz9lOULkOcKmHpOGGrGQtDg8BeWQF44WOCyJZAMYDz/X5yCo1ptt+lDJLc/xBp/agX6A7yTQIMBwBYcbX352H1p8cfWJ9ZltZ0PcWvo9thrLJykCL/cyIEE1drYpkCWl2cgM1VOHf1itqfb8CBU4Y099Xic5nujMsFhfYTIUf7VFCcvPViO0ce4EihKA2G2ptL5/waYIV7fGLF+OZxAmc= anokhimehta@Anokhis-Air.lan

# Anurag  
# ssh-rsa AAAA...

# Dhanush
ssh-rsa AAAAB3NzaC1yc2EAAAADAQABAAABgQDnttdrVF51qay5h3qVBIQ6oMDr+jNlrslZD76agx/H2R5zMxNqOnLDJyL4SR242M+YhLRDrn8KF2tBzHBPFHAA8OWgO8FoXopa/qY7mQjealusbSr4uGWxXJV5XZ5L6iY3DAURpUdgC+AY0RHqejd4KckXBKUs6ZOAScT3sFFC3C+m0WUm3jYB2iw2iSivevde1RxoXdQ2F9FBvz1c5DNL9Q2WPg0C2QjUTYjDw6nnCIjmx7vUedUFN/Wjxpn/l1DN4pVOhkoj7uHVwG6hMs0WMx9J0CZ5FvX+VBAKldKx87WCVKQ8rrxTw33/vEUIN/6rxOW/dwDctmEgvu7oP3cMzf+mvPQnNCbUYAjmvFbkJ8iKgZXj66uJvuzbzzHfVce7/BJMto5EKGSHBxoqZshQL4IN4nAP7LLuV7X6tVbPo9XO+WUvZGa2fIMTM8nDFG54n/TbMiew1THA8gAob6lVgRsrlRYmoQJK1HpRitsvq6vokTc3MopAsLY5g7W7UGM= dhanu@acer

EOF
""")
print("Team SSH keys added!")

## 6. a. (Run if repo is NOT in Jupyter Interface) Clone your repo onto the node

In [ ]:
TOKEN = "ghp_xxxx" # private repo required a token, public repo can be cloned without it

s.execute(f"git clone https://{TOKEN}@github.com/anokhimehta/bestshot.git")

## 6. b. (Run this if repo is already in Jupyter Interface) Go into project folder and pull any changes

In [ ]:
s.execute("cd bestshot && git pull")

## 7. Install Docker

In [ ]:
s.execute("curl -sSL https://get.docker.com/ | sudo sh")
s.execute("sudo groupadd -f docker; sudo usermod -aG docker $USER")

## 8. Verify the GPU is visible

In [ ]:
s.execute("rocm-smi")

## 9. Build your serving Docker image

In [ ]:
REPO_DIR = "bestshot/serving"    # using the folder name git clone created (bestshot), and inside where the Dockerfile is (serving)

s.execute(f"docker build -t bestshot-serve -f {REPO_DIR}/docker/Dockerfile.serve {REPO_DIR}/")

## 10. Start the serving container

In [ ]:
s.execute(
    "docker run -d --rm "
    "--network host "
    "--device=/dev/kfd "
    "--device=/dev/dri "
    "--group-add video "
    "--ipc=host "
    "--name bestshot-api "
    "bestshot-serve"
)

## 11. Build training container

In [ ]:
s.execute("docker build -t bestshot-train:latest -f ~/bestshot/training/docker/Dockerfile ~/bestshot/")

## 12.(Optional) Trigger training manually 

In [ ]:
s.execute("""docker run --rm \\
  --network host \\
  --env-file ~/bestshot/serving/.env \\
  --device=/dev/kfd --device=/dev/dri \\
  --group-add video \\
  --shm-size=8g \\
  ghcr.io/anokhimehta/bestshot-training:latest \\
  python3 training/train.py --config training/config/partial_finetune_highlr.yaml""")

## 13. Options for running the code:
1. running on terminal 
2. running on vs code

### Option 1: Terminal - SSH in to run benchmarks manually

From your **local terminal**, run:

```bash
ssh -i ~/.ssh/id_rsa_chameleon cc@<FLOATING_IP>
```

Then on the node:

1. Pull latest code
```bash
    cd bestshot
    git pull
```

2. Rebuild the image with your changes
```bash
    docker build -t bestshot-serve -f serving/docker/Dockerfile.serve serving/
```

3. Clean up old container
```bash
    docker rm -f bestshot-api
```

4. start the service 
```bash
    cd ~/bestshot/serving
    chmod +x start.sh
    ./start.sh
```

### Option 2: VS Code

Run the command below first and then follow the rest of the steps

In [ ]:
# Add ability to run commands through run.sh
# add the "executable" permission, which tells the OS "this file is allowed to be run as a script."
s.execute("chmod +x bestshot/serving/start.sh")

# terminal equivalent: chmod +x serving/start.sh

1. Connect to ssh from VS Code:

    * Press Cmd + Shift + P
    * Type: Remote-SSH: Connect to Host
    * Select: chameleon-node

    Then, VS Code will:
    * SSH into the machine
    * Open a remote workspace
    * Let you edit files as if they’re local


2. To set up env,  open terminal in VS Code:

    Once connected open a new terminal and run:
    ```bash 
        code . 
    ```
     Now you’re running commands on the remote machine, not your laptop. And you are able to see your repo on the application.

3. Set up the node: 

    a. Confirm you are in 
    ```bash
        cc@node-bestshot-lg3323-nyu-edu:~/bestshot$
    ```

    b. Pull latest code
    ```bash
        git pull
    ```

    c. Make a .env file and fill it with credentials
    ```bash
        cd ~/bestshot/serving
        nano .env
    ```
        credentials template:
    ```bash
        # Immich
        IMMICH_URL=http://<floating-ip>:30283
        IMMICH_API_KEY=<your-immich-api-key>

        # Serving
        SERVING_URL=http://localhost:8000
        POLL_INTERVAL=30

        # MLflow
        MLFLOW_TRACKING_URI=http://129.114.25.185:8000

        # OpenStack / Swift
        OS_AUTH_URL=https://chi.tacc.chameleoncloud.org:5000/v3
        OS_AUTH_TYPE=v3applicationcredential
        OS_APPLICATION_CREDENTIAL_ID=<your-id>
        OS_APPLICATION_CREDENTIAL_SECRET=<your-secret>
        OS_REGION_NAME=CHI@TACC
        BUCKET_NAME=ak12754-data-proj19
    ```
    to get Immich API Key:

        1. go to http://<floating-ip>:30283 in browser 

        2. make account

        3. profile icon >> account settings >> API Keys

    d. Rebuild Docker image with changes (e.g. change: Python files, dependencier, Dockerfile):
    ```bash
        docker build -t bestshot-serve -f docker/Dockerfile.serve .
    ```

    f. Start API and run benchmark, examples:
    ```bash
        ./start.sh
    ```





## 14. Startup Script to bring everything up

In [ ]:
# NOTE: BEFORE RUNNING THE BELOW CELL, MAKE SURE TO UPDATE THE .env FILE WITH THE FLOATING IP ADDRESS OF YOUR NODE (FIP) AND ANY OTHER RELEVANT CONFIGS
# Start all BestShot services
s.execute("cd ~/bestshot/serving && ./start.sh")
print("All services started!")
print(f"  API:        http://{fip}:8000/health")
print(f"  Grafana:    http://{fip}:3000")
print(f"  Prometheus: http://{fip}:9090")
print(f"  Immich:     http://{fip}:30283")


## 15. Delete Resources

In [ ]:
# Stop the API container if it's still running
s.execute("docker stop bestshot-api")

In [ ]:
# Delete the container 
s.execute("docker rm bestshot-api")

In [ ]:
# Shut down the bare metal server to release it back
s.delete()